In [ ]:
import os, json, time, hashlib
import numpy as np, pandas as pd, requests

OUT = "oa_out"; CACHE = os.path.join(OUT, "cache"); BASE = "https://api.openalex.org"
MAILTO = "<REDACTED_EMAIL>"; OA_KEY = os.environ.get("OPENALEX_API_KEY", "")
OA_KEY = "<REDACTED_API_KEY>"
BATCH = 50; NCALL = 0
os.makedirs(CACHE, exist_ok=True)

def oa_get(path, params, tries=8):
    global NCALL
    fn = os.path.join(CACHE, hashlib.md5((path + json.dumps(params, sort_keys=True)).encode()).hexdigest() + ".json")
    if os.path.exists(fn):
        try: return json.load(open(fn, encoding="utf-8"))
        except json.JSONDecodeError: os.remove(fn)
    assert OA_KEY, "환경변수 OPENALEX_API_KEY를 설정하세요"
    p = dict(params); p["mailto"] = MAILTO; p["api_key"] = OA_KEY; last = None
    for t in range(tries):
        try: r = requests.get(BASE + path, params=p, timeout=120)
        except requests.RequestException as e:
            last = str(e); time.sleep(min(60, 2 ** t)); continue
        if r.status_code == 200:
            NCALL += 1; j = r.json(); json.dump(j, open(fn, "w", encoding="utf-8"), ensure_ascii=False); time.sleep(0.2); return j
        last = f"HTTP {r.status_code}: {r.text[:150]}"
        if r.status_code in (429, 500, 502, 503, 504): time.sleep(min(90, 3 * 2 ** t)); continue
        raise RuntimeError(last)
    raise RuntimeError("재시도 초과: " + str(last))

sid = lambda x: (x or "").split("/")[-1]
L = pd.read_csv(f"{OUT}/collab_links.csv")
M = pd.read_csv("oa_inst_map.csv", dtype=str).fillna("")
M = M[M["openalex_ids"] != ""]
firm_ids = sorted({i for s in M["openalex_ids"] for i in s.split("|")})
ids = sorted(set(L["inst_id"].dropna().astype(str)) | set(firm_ids))
print(f"조회 대상 기관 {len(ids):,}곳 -> 예상 호출 {int(np.ceil(len(ids)/BATCH))}회")

rec = []
for k in range(0, len(ids), BATCH):
    b = ids[k:k + BATCH]
    j = oa_get("/institutions", {"filter": "openalex:" + "|".join(b), "per_page": BATCH,
                                 "select": "id,display_name,type,country_code,geo"})
    for x in j.get("results", []):
        g = x.get("geo") or {}
        rec.append(dict(inst_id=sid(x["id"]), name=x.get("display_name"), type=x.get("type"),
                        country=x.get("country_code"), city=g.get("city"), region=g.get("region"),
                        lat=g.get("latitude"), lon=g.get("longitude")))
G = pd.DataFrame(rec).drop_duplicates("inst_id")
G.to_csv(f"{OUT}/inst_geo.csv", index=False, encoding="utf-8-sig")
print("저장 inst_geo.csv", G.shape, "| 좌표 결측", int(G["lat"].isna().sum()), "| 호출", NCALL)

In [ ]:
SITES = [
 # corp_norm, 거점명, 질의 주소, 폴백(시군), 전 기간 존재, 출처
 ("トヨタ自動車","본사(도요타시)","愛知県豊田市トヨタ町1","愛知県豊田市",True,"본사 주소는 지식 기반, 확인 필요"),
 ("トヨタ自動車","히가시후지연구소","静岡県裾野市","静岡県裾野市",True,"검색: 뉴스"),
 ("トヨタ自動車","시모야마 기술센터","愛知県豊田市","愛知県岡崎市",False,"검색: 2024 전면 개업"),
 ("三菱電機","선단기술종합연구소(아마가사키)","兵庫県尼崎市塚口本町8-1-1","兵庫県尼崎市",True,"검색: 공식"),
 ("三菱電機","선단기술종합연구소(히메지)","兵庫県姫路市","兵庫県姫路市",True,"검색: 공식"),
 ("三菱電機","정보기술종합연구소(오후나)","神奈川県鎌倉市大船","神奈川県鎌倉市",True,"검색: 공식"),
 ("日産自動車","본사","神奈川県横浜市西区高島1-1-1","神奈川県横浜市西区",True,"검색: 공식"),
 ("日産自動車","종합연구소(오이)","神奈川県横須賀市夏島町1","神奈川県横須賀市",True,"검색: 공식"),
 ("日産自動車","테크니컬센터","神奈川県厚木市岡津古久560-2","神奈川県厚木市",True,"검색: 현 자료"),
 ("日産自動車","선진기술개발센터","神奈川県厚木市森の里青山1-1","神奈川県厚木市",True,"검색: 공식"),
 ("ホンダ","와코","埼玉県和光市中央1-4-1","埼玉県和光市",True,"검색: 공식"),
 ("ホンダ","아사카","埼玉県朝霞市泉水3-15-1","埼玉県朝霞市",True,"검색: 공식"),
 ("ホンダ","도치기(하가)","栃木県芳賀郡芳賀町大字下高根沢4630","栃木県芳賀郡芳賀町",True,"검색: 공시"),
 ("日本電信電話","무사시노","東京都武蔵野市","東京都武蔵野市",True,"검색: 총무성 자료"),
 ("日本電信電話","요코스카","神奈川県横須賀市","神奈川県横須賀市",True,"검색: 총무성 자료"),
 ("日本電信電話","아쓰기","神奈川県厚木市","神奈川県厚木市",True,"검색: 총무성 자료"),
 ("日本電信電話","쓰쿠바","茨城県つくば市","茨城県つくば市",True,"검색: 총무성 자료"),
 ("日本電信電話","게이한나","京都府相楽郡","京都府木津川市",True,"검색: 총무성 자료"),
]
S = pd.DataFrame(SITES, columns=["corp_norm","site","q","q_fb","all_period","source"])

# 직접 채울 기업은 아래 CSV에 같은 열로 추가(코드가 합쳐 줌)
extra = "firm_sites_extra.csv"
if os.path.exists(extra):
    S = pd.concat([S, pd.read_csv(extra)], ignore_index=True)

GC = os.path.join(OUT, "site_geocache.json")
cache = json.load(open(GC, encoding="utf-8")) if os.path.exists(GC) else {}
def geocode(q):
    if q in cache: return cache[q]
    r = requests.get("https://nominatim.openstreetmap.org/search",
                     params={"q": q, "format": "json", "limit": 1, "countrycodes": "jp"},
                     headers={"User-Agent": f"gx-research/1.0 ({MAILTO})"}, timeout=60)
    time.sleep(1.2)
    res = r.json(); cache[q] = (float(res[0]["lat"]), float(res[0]["lon"])) if res else None
    json.dump(cache, open(GC, "w", encoding="utf-8"), ensure_ascii=False); return cache[q]

lat, lon, lvl = [], [], []
for _, r in S.iterrows():
    c = geocode(r["q"]); lv = "주소"
    if c is None: c = geocode(r["q_fb"]); lv = "시군"
    lat.append(c[0] if c else np.nan); lon.append(c[1] if c else np.nan); lvl.append(lv if c else "실패")
S["lat"], S["lon"], S["geo_level"] = lat, lon, lvl
S.to_csv(f"{OUT}/firm_sites_geo.csv", index=False, encoding="utf-8-sig")
print(S[["corp_norm","site","geo_level","lat","lon"]].round(3).to_string(index=False))

# 아직 확인하지 않은 기업 목록 (이 기업들은 OpenAlex 본사 좌표로 대체됨)
done = set(S["corp_norm"]); todo = M[~M["corp_norm"].isin(done)][["corp_norm", "corp_ja"]] if "corp_ja" in M else M[~M["corp_norm"].isin(done)][["corp_norm"]]
todo.to_csv(f"{OUT}/firm_sites_todo.csv", index=False, encoding="utf-8-sig")
print(f"\n거점 미확인 기업 {len(todo)}곳 -> firm_sites_todo.csv (OpenAlex 본사 좌표로 대체)")

In [ ]:
G = pd.read_csv(f"{OUT}/inst_geo.csv"); S = pd.read_csv(f"{OUT}/firm_sites_geo.csv")
L = pd.read_csv(f"{OUT}/collab_links.csv")
YMIN = int(L["year"].min()); BLK = 5; L["blk"] = (L["year"] - YMIN) // BLK
LAB = {b: f"{YMIN+b*BLK}-{YMIN+b*BLK+BLK-1}" for b in sorted(L["blk"].unique())}

def hav(lat1, lon1, lat2, lon2):
    p = np.pi / 180; a = (np.sin((lat2-lat1)*p/2)**2 + np.cos(lat1*p)*np.cos(lat2*p)*np.sin((lon2-lon1)*p/2)**2)
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

# 기업 본사(OpenAlex): 기업별 첫 번째 기관 ID의 geo. 여러 ID가 있으면 아래 출력으로 직접 확인
M["first_id"] = M["openalex_ids"].str.split("|").str[0]
HQ = M[["corp_norm", "first_id"]].merge(G.rename(columns={"inst_id": "first_id"})[["first_id","name","city","lat","lon"]], on="first_id", how="left")
HQ = HQ.rename(columns={"lat": "hq_lat", "lon": "hq_lon", "city": "hq_city", "name": "hq_name"})
print("본사 좌표 결측 기업:", int(HQ["hq_lat"].isna().sum()))
print(HQ[HQ["corp_norm"].isin(S["corp_norm"])][["corp_norm", "hq_name", "hq_city"]].to_string(index=False))  # 거점 확인 기업의 매칭 점검

# 링크 x 거점 -> 최소 거리 (전 기간 존재 거점만)
S1 = S[(S["all_period"]) & S["lat"].notna()][["corp_norm", "site", "lat", "lon"]]
P = L[["work_id", "corp_norm", "inst_id", "year", "blk", "country", "itype", "geo"]].merge(
    G.rename(columns={"inst_id":"inst_id", "lat":"p_lat", "lon":"p_lon", "region":"p_region", "city":"p_city"})[["inst_id","p_lat","p_lon","p_region","p_city"]],
    on="inst_id", how="left")
P = P.merge(HQ[["corp_norm", "hq_lat", "hq_lon"]], on="corp_norm", how="left")
P["d_hq"] = hav(P["hq_lat"], P["hq_lon"], P["p_lat"], P["p_lon"])

pm = P[["work_id","corp_norm","inst_id","p_lat","p_lon"]].merge(S1, on="corp_norm")
pm["d"] = hav(pm["lat"], pm["lon"], pm["p_lat"], pm["p_lon"])
dmin = pm.groupby(["work_id","corp_norm","inst_id"])["d"].min().rename("d_site").reset_index()
P = P.merge(dmin, on=["work_id","corp_norm","inst_id"], how="left")
P["d_min"] = P["d_site"].fillna(P["d_hq"])                       # 거점 미확인 기업은 본사 거리로 대체
P["d_source"] = np.where(P["d_site"].notna(), "거점", "본사(대체)")
P["ln_d"] = np.log1p(P["d_min"]); P["ln_d_hq"] = np.log1p(P["d_hq"])

# 신규/재협력(기존 정의와 동일: 기업-기관 첫 협력 연도, 첫 블록 이후만 신규)
fy = P.groupby(["corp_norm", "inst_id"])["year"].min().rename("first_y").reset_index()
P = P.merge(fy, on=["corp_norm", "inst_id"]); Y0 = YMIN + BLK
P["new"] = (P["year"] == P["first_y"]) & (P["first_y"] >= Y0)
P["rel"] = np.where(P["new"], "신규", np.where(P["year"] > P["first_y"], "재협력", "첫블록(구분불가)"))
P.to_csv(f"{OUT}/collab_links_geo.csv", index=False, encoding="utf-8-sig")

# ---- 기술통계: 거리 감쇠의 시기별 변화 (S1 사전 점검), 신규 vs 재협력 (S2 사전 점검)
print("\n거리 산출 출처:", P["d_source"].value_counts().to_dict(), "| 좌표 결측 링크:", int(P["d_min"].isna().sum()))
print("\n[S1] 5년 블록별 협력 거리(km) 중앙값 / 평균 ln(1+d): 국내(JP) vs 해외")
P["area"] = np.where(P["country"] == "JP", "국내", "해외")
print(P.pivot_table(index="area", columns="blk", values="d_min", aggfunc="median").rename(columns=LAB).round(0).to_string())
print(P.pivot_table(index="area", columns="blk", values="ln_d", aggfunc="mean").rename(columns=LAB).round(2).to_string())
print("\n[S2] 신규 vs 재협력 링크의 거리(km, 중앙값 / ln 평균 / 건수)")
print(P[P["rel"] != "첫블록(구분불가)"].groupby(["area", "rel"]).agg(중앙값=("d_min","median"), ln평균=("ln_d","mean"), n=("d_min","size")).round(2).to_string())
print("\n[민감도] 거점 거리 vs 본사 거리 상관(거점 확인 기업만):", round(P[P["d_source"]=="거점"][["ln_d","ln_d_hq"]].corr().iloc[0,1], 3))
print("\n[국내 파트너 도도부현 상위]"); print(P[P["area"]=="국내"]["p_region"].value_counts().head(10).to_string())

In [ ]:
#

In [ ]:
import statsmodels.formula.api as smf
pd.set_option("display.width", 220)

P["area"] = np.where(P["country"] == "JP", "국내", "해외")
Dm = P[P["area"] == "국내"].copy()

# ---- 1. 국내만 본 거점 vs 본사 거리 상관, 출처별 추세 (출처 편향 점검)
c = Dm[Dm["d_source"] == "거점"][["ln_d", "ln_d_hq"]].corr().iloc[0, 1]
print(f"[1a] 국내 거점-본사 거리 상관: {c:.3f}")
print("[1b] 국내 ln 거리 평균: 블록 x 거리 출처")
print(Dm.pivot_table(index="d_source", columns="blk", values="ln_d", aggfunc="mean").rename(columns=LAB).round(2).to_string())
print("[1c] 블록별 '거점' 출처 링크 비중:", Dm.groupby("blk")["d_source"].apply(lambda s: (s == "거점").mean()).round(3).rename(index=LAB).to_dict())

# ---- 2. 국내 거리 증가는 구성 효과인가: 파트너 지역·유형 분해
Dm["reg"] = Dm["p_region"].where(Dm["p_region"].isin(Dm["p_region"].value_counts().head(8).index), "기타")
print("\n[2a] 국내 파트너 지역 비중(블록 내)")
print(pd.crosstab(Dm["reg"], Dm["blk"], normalize="columns").rename(columns=LAB).round(3).to_string())
print("[2b] 지역별 평균 ln 거리(전체 기간)와 이를 고정해 계산한 구성 예측 평균 vs 실제")
mu = Dm.groupby("reg")["ln_d"].mean()
sh = pd.crosstab(Dm["reg"], Dm["blk"], normalize="columns")
pred = (sh.mul(mu, axis=0)).sum(); act = Dm.groupby("blk")["ln_d"].mean()
print(pd.DataFrame({"실제": act, "구성만으로 예측": pred}).rename(index=LAB).round(3).to_string())
print("[2c] 파트너 유형 비중(블록 내)")
print(pd.crosstab(Dm["itype"], Dm["blk"], normalize="columns").rename(columns=LAB).round(3).to_string())

# ---- 3. 해외: 국가 구성만으로 거리 변화가 설명되는가
Fo = P[P["area"] == "해외"].copy()
Fo["ctry"] = Fo["country"].where(Fo["country"].isin(Fo["country"].value_counts().head(10).index), "기타")
muc = Fo.groupby("ctry")["ln_d"].mean(); shc = pd.crosstab(Fo["ctry"], Fo["blk"], normalize="columns")
print("\n[3] 해외 ln 거리: 실제 vs 국가 구성만으로 예측")
print(pd.DataFrame({"실제": Fo.groupby("blk")["ln_d"].mean(), "구성만으로 예측": shc.mul(muc, axis=0).sum()}).rename(index=LAB).round(3).to_string())

# ---- 4. 통제 모형 (dyad-블록 단위, 기업 고정효과, 기업 군집 표준오차)
def dyad_frame(d):
    x = d.sort_values("d_min").drop_duplicates(["corp_norm", "inst_id", "blk"])    # 같은 기업-기관-블록은 1행
    x["new_d"] = ((x["first_y"] >= Y0) & ((x["first_y"] - YMIN) // BLK == x["blk"])).astype(int)
    return x
def reg(d, f, title):
    d = d.dropna(subset=["ln_d"]); g = pd.factorize(d["corp_norm"])[0]
    m = smf.ols(f, data=d).fit(cov_type="cluster", cov_kwds={"groups": g})
    keep = [v for v in m.params.index if not v.startswith("C(") and v != "Intercept"] or list(m.params.index[1:4])
    print(f"\n{title} | N={int(m.nobs)}, 기업 {d['corp_norm'].nunique()}곳")
    print(pd.DataFrame({"coef": m.params[keep], "se": m.bse[keep], "p": m.pvalues[keep]}).round(3).to_string())
    return m

Dd = dyad_frame(Dm)
Dd["blk_c"] = Dd["blk"]
reg(Dd, "ln_d ~ C(blk) + C(itype) + C(corp_norm)", "[4a] 국내 거리: 블록 효과(기준=첫 블록), 파트너 유형·기업 통제")
m = reg(Dd, "ln_d ~ C(blk) + C(itype) + C(p_region) + C(corp_norm)", "[4b] 위 + 파트너 지역 통제(블록 효과가 남는지)")
print("  (4a/4b 블록 계수는 아래 별도 출력)"); print(m.params.filter(like="C(blk)").round(3).to_string())
Dd2 = Dd[Dd["d_source"] == "거점"]
reg(Dd2, "ln_d ~ C(blk) + C(itype) + C(corp_norm)", "[4c] 거점 확인 기업만")
reg(Dd, "ln_d ~ new_d + C(blk) + C(itype) + C(corp_norm)", "[4d] S2 국내: 신규 dyad 여부 (new_d, 기준=그 외)")
reg(Dd[Dd["first_y"] >= Y0], "ln_d ~ new_d + C(blk) + C(itype) + C(corp_norm)", "[4e] S2 국내: 첫 블록 이후 시작 dyad만(신규 vs 같은 dyad 재협력)")
Fd = dyad_frame(Fo)
reg(Fd, "ln_d ~ new_d + C(blk) + C(itype) + C(ctry) + C(corp_norm)", "[4f] S2 해외: 국가 고정효과 포함 신규 dyad 여부")

In [ ]:
#

In [ ]:
import statsmodels.formula.api as smf
pd.set_option("display.width", 220)

print("국내 링크 중 p_region 결측:", int(Dm["p_region"].isna().sum()), "/", len(Dm))
Dm["p_region"] = Dm["p_region"].fillna("미상")

def reg(d, f, title, show=r"^(C\(blk\)|new_d)"):
    mod = smf.ols(f, data=d, missing="drop")
    idx = mod.data.row_labels                                   # 모형이 실제로 쓴 행
    g = pd.factorize(d.loc[idx, "corp_norm"])[0]
    m = mod.fit(cov_type="cluster", cov_kwds={"groups": g})
    keep = [v for v in m.params.index if pd.Series(v).str.contains(show).iloc[0]]
    print(f"\n{title} | N={int(m.nobs)}, 기업 {d.loc[idx, 'corp_norm'].nunique()}곳")
    print(pd.DataFrame({"coef": m.params[keep], "se": m.bse[keep], "p": m.pvalues[keep]}).round(3).to_string())
    return m

def dyad_frame(d):
    x = d.sort_values("d_min").drop_duplicates(["corp_norm", "inst_id", "blk"]).copy()
    x["new_d"] = ((x["first_y"] >= Y0) & ((x["first_y"] - YMIN) // BLK == x["blk"])).astype(int)
    return x

Dd = dyad_frame(Dm)
reg(Dd, "ln_d ~ C(blk) + C(itype) + C(p_region) + C(corp_norm)", "[4b] 파트너 지역 통제 추가")
reg(Dd[Dd["d_source"] == "거점"], "ln_d ~ C(blk) + C(itype) + C(corp_norm)", "[4c] 거점 확인 기업만")
reg(Dd, "ln_d ~ new_d + C(blk) + C(itype) + C(corp_norm)", "[4d] S2 국내: 신규 dyad 여부")
reg(Dd[Dd["first_y"] >= Y0], "ln_d ~ new_d + C(blk) + C(itype) + C(corp_norm)", "[4e] S2 국내: 첫 블록 이후 시작 dyad만")

Fo = P[P["country"] != "JP"].copy()
Fo["ctry"] = Fo["country"].where(Fo["country"].isin(Fo["country"].value_counts().head(10).index), "기타")
Fd = dyad_frame(Fo)
reg(Fd, "ln_d ~ new_d + C(blk) + C(itype) + C(ctry) + C(corp_norm)", "[4f] S2 해외: 국가 통제 후 신규 dyad 여부")

In [ ]:
import numpy as np, pandas as pd, statsmodels.formula.api as smf
pd.set_option("display.width", 220)

YEND = int(P["year"].max()) - 1; H = 3
Pd = P[P["country"] == "JP"].copy(); Pd["p_region"] = Pd["p_region"].fillna("미상")
dy = (Pd.groupby(["corp_norm", "inst_id"])
        .agg(first_y=("year", "min"), itype=("itype", "first"), region=("p_region", "first"),
             ln_d=("ln_d", "mean"), d_source=("d_source", "first"), n_links=("work_id", "size")).reset_index())
yrs = Pd.groupby(["corp_norm", "inst_id"])["year"].apply(lambda s: sorted(set(s)))
dy["reco"] = [int(any((y > f) and (y <= f + H) for y in yrs[(c, i)])) for c, i, f in zip(dy["corp_norm"], dy["inst_id"], dy["first_y"])]
el = dy[(dy["first_y"] >= Y0) & (dy["first_y"] <= YEND - H)].copy()
el["ln_d_c"] = el["ln_d"] - el["ln_d"].mean()
el["reg_g"] = el["region"].where(el["region"].isin(el["region"].value_counts().head(10).index), "기타")
print(f"[A0] 신규 dyad {len(el):,} | H={H}년 내 재협력률 {el['reco'].mean():.3f}")

# ---- 진단: 왜 분리가 생겼는가
fs = el.groupby("corp_norm")["reco"].agg(["size", "mean"])
print(f"[A0b] 기업별 dyad 수 중앙값 {fs['size'].median():.0f} | dyad 5개 미만 기업 {int((fs['size'] < 5).sum())}곳 | 재협력이 전부 0 또는 전부 1인 기업 {int(((fs['mean'] == 0) | (fs['mean'] == 1)).sum())}곳")
el["d_bin"] = pd.qcut(el["ln_d"], 4, duplicates="drop")
print("[A0c] 거리 사분위별 재협력률 (기술통계)"); print(el.groupby("d_bin", observed=True)["reco"].agg(["mean", "size"]).round(3).to_string())

def fit_ols(d, f, title):
    mod = smf.ols(f, data=d, missing="drop"); idx = mod.data.row_labels
    m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[idx, "corp_norm"])[0]})
    keep = [v for v in m.params.index if v == "ln_d_c"]
    print(f"\n{title} | N={int(m.nobs)}, 기업 {d.loc[idx, 'corp_norm'].nunique()}곳")
    print(pd.DataFrame({"coef": m.params[keep], "se": m.bse[keep], "p": m.pvalues[keep]}).round(4).to_string())
fit_ols(el, "reco ~ ln_d_c + C(itype) + C(region) + C(corp_norm)", "[A1] LPM: 거리 -> 재협력 (유형·지역·기업 FE)")
fit_ols(el, "reco ~ ln_d_c + C(itype) + C(corp_norm)", "[A2] LPM: 지역 FE 없이")
fit_ols(el, "reco ~ ln_d_c + C(itype) + C(reg_g)", "[A3] LPM: 기업 FE 없이(상위 지역만)")
fit_ols(el[el["d_source"] == "거점"], "reco ~ ln_d_c + C(itype)", "[A4] 거점 확인 기업만 (참고용, 기업 수 적음)")

# ---- 보조: 로짓(고정효과 축소). 수렴 실패 시 건너뜀
try:
    mod = smf.logit("reco ~ ln_d_c + C(itype) + C(reg_g)", data=el, missing="drop"); idx = mod.data.row_labels
    m = mod.fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(el.loc[idx, "corp_norm"])[0]})
    print(f"\n[A5] 로짓(기업 FE 없이) | N={int(m.nobs)} | ln_d_c OR={np.exp(m.params['ln_d_c']):.3f}, p={m.pvalues['ln_d_c']:.3f}")
except Exception as e:
    print("\n[A5] 로짓 생략:", str(e)[:100])

In [ ]:
import numpy as np, pandas as pd
pd.set_option("display.width", 220)
try:
    import libpysal as lps, esda, spreg
    from scipy import sparse
except ImportError as e:
    raise SystemExit("pip install libpysal esda spreg 후 다시 실행: " + str(e))

nw = Pd[(Pd["year"] == Pd["first_y"]) & (Pd["first_y"] >= Y0) & (Pd["p_region"] != "미상")]
cnt = nw.groupby(["p_region", "blk"]).agg(n_new=("inst_id", "size"), n_firms=("corp_norm", "nunique")).reset_index()

GJ = G[(G["country"] == "JP") & G["region"].notna()]
RA = GJ.groupby("region").agg(lat=("lat", "mean"), lon=("lon", "mean"), n_inst=("inst_id", "nunique"),
                              n_fac=("type", lambda s: (s == "facility").sum())).reset_index()
regs = sorted(set(RA["region"]) & set(nw["p_region"]))
RA = RA.set_index("region").loc[regs]
print(f"[B0] 지역 수 {len(regs)} | 신규 협력 수 상위 지역:"); print(nw["p_region"].value_counts().head(8).to_string())

def hav_m(lat, lon):
    p = np.pi / 180; la, lo = lat * p, lon * p
    a = np.sin((la[:, None] - la[None, :]) / 2) ** 2 + np.cos(la)[:, None] * np.cos(la)[None, :] * np.sin((lo[:, None] - lo[None, :]) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))
Dmat = hav_m(RA["lat"].values, RA["lon"].values)
def knn_w(k):
    nb = {i: [int(j) for j in np.argsort(Dmat[i])[1:k + 1]] for i in range(len(regs))}
    w = lps.weights.W(nb, {i: [1.0] * k for i in nb}); w.transform = "R"; return w

blks = sorted(cnt["blk"].unique())
Y = cnt.pivot(index="p_region", columns="blk", values="n_new").reindex(regs).reindex(columns=blks).fillna(0)
print("\n[B1] 블록별 Moran's I (ln(1+신규 협력 수))")
rows = []
for k in (4, 6):
    w = knn_w(k)
    for b in blks:
        mi = esda.Moran(np.log1p(Y[b].values), w, permutations=999)
        rows.append(dict(블록=LAB[b], k=k, I=round(mi.I, 3), p_sim=round(mi.p_sim, 3)))
print(pd.DataFrame(rows).to_string(index=False))

long = (pd.MultiIndex.from_product([blks, regs], names=["blk", "p_region"]).to_frame(index=False)
          .merge(cnt, on=["p_region", "blk"], how="left").fillna({"n_new": 0, "n_firms": 0}))
long["ln_n"] = np.log1p(long["n_new"])
long = long.join(RA[["n_inst", "n_fac"]], on="p_region")
long["l_n_inst"] = np.log1p(long["n_inst"]); long["l_n_fac"] = np.log1p(long["n_fac"])
long = long.sort_values(["blk", "p_region"]).reset_index(drop=True)       # 블록 안에서 regs 순서와 일치
X = pd.concat([long[["l_n_inst", "l_n_fac"]], pd.get_dummies(long["blk"], prefix="b", drop_first=True).astype(float)], axis=1)
y = long[["ln_n"]].values
for k in (4, 6):
    w = knn_w(k); WS = sparse.block_diag([w.sparse for _ in blks], format="csr")
    WB = lps.weights.WSP2W(lps.weights.WSP(WS)); WB.transform = "R"
    ols = spreg.OLS(y, X.values, w=WB, spat_diag=True, moran=True, name_x=list(X.columns), name_y="ln_n")
    print(f"\n[B2] k={k}  OLS 진단: Moran 잔차 p={ols.moran_res[2]:.3f} | LM-lag p={ols.lm_lag[1]:.3f} | LM-error p={ols.lm_error[1]:.3f}")
    m = spreg.ML_Lag(y, X.values, w=WB, name_x=list(X.columns), name_y="ln_n")
    out = pd.DataFrame({"var": m.name_x, "coef": m.betas.ravel(), "p": [z[1] for z in m.z_stat]})
    print(f"SAR: N={len(long)}, rho={m.rho:.3f}")
    print(out.round(3).to_string(index=False))

In [ ]:
# 지역 48개 중 이상한 라벨 확인 + 연결 안 된 성분 확인
print(sorted(regs))
w = knn_w(4)
import libpysal as lps
comp = lps.weights.util.get_ids  # (참고용)
from scipy.sparse.csgraph import connected_components
n_comp, lab = connected_components(w.sparse, directed=False)
for c in range(n_comp):
    print(c, [regs[i] for i in np.where(lab == c)[0]])

In [ ]:
import unicodedata, numpy as np, pandas as pd, warnings
import libpysal as lps, esda, spreg
from scipy import sparse
from scipy.sparse.csgraph import connected_components
warnings.filterwarnings("ignore", message="The weights matrix is not fully connected")
pd.set_option("display.width", 220)

def norm(s):
    if pd.isna(s): return s
    return "".join(c for c in unicodedata.normalize("NFKD", str(s)) if not unicodedata.combining(c))
Gn = G.copy(); Gn["region"] = Gn["region"].map(norm)
Pn = Pd.copy(); Pn["p_region"] = Pn["p_region"].map(norm)

nw = Pn[(Pn["year"] == Pn["first_y"]) & (Pn["first_y"] >= Y0) & (Pn["p_region"] != "미상") & Pn["p_region"].notna()]
cnt = nw.groupby(["p_region", "blk"]).agg(n_new=("inst_id", "size")).reset_index()
GJ = Gn[(Gn["country"] == "JP") & Gn["region"].notna()]
RA = GJ.groupby("region").agg(lat=("lat", "mean"), lon=("lon", "mean"), n_inst=("inst_id", "nunique"),
                              n_fac=("type", lambda s: (s == "facility").sum())).reset_index()
regs = sorted(set(RA["region"]) & set(nw["p_region"])); RA = RA.set_index("region").loc[regs]
print(f"[B0] 지역 수 {len(regs)} (47이어야 함)")
print("  신규 협력 상위:", nw["p_region"].value_counts().head(6).to_dict())

def hav_m(lat, lon):
    p = np.pi / 180; la, lo = lat * p, lon * p
    a = np.sin((la[:, None] - la[None, :]) / 2) ** 2 + np.cos(la)[:, None] * np.cos(la)[None, :] * np.sin((lo[:, None] - lo[None, :]) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))
Dmat = hav_m(RA["lat"].values, RA["lon"].values)
off = Dmat + np.eye(len(regs)) * 1e9
print(f"  최근접 지역 간 최소 거리 {off.min():.1f}km (0에 가까우면 중복 좌표)")

def knn_w(k):
    nb = {i: [int(j) for j in np.argsort(Dmat[i])[1:k + 1]] for i in range(len(regs))}
    w = lps.weights.W(nb, {i: [1.0] * k for i in nb}); w.transform = "R"; return w
print("  가중행렬 연결 성분(k=4):", connected_components(knn_w(4).sparse, directed=False)[0])

blks = sorted(cnt["blk"].unique())
Y = cnt.pivot(index="p_region", columns="blk", values="n_new").reindex(regs).reindex(columns=blks).fillna(0)
rows = []
for k in (4, 6):
    w = knn_w(k)
    for b in blks:
        mi = esda.Moran(np.log1p(Y[b].values), w, permutations=999)
        rows.append(dict(블록=LAB[b], k=k, I=round(mi.I, 3), p_sim=round(mi.p_sim, 3)))
print("\n[B1] Moran's I"); print(pd.DataFrame(rows).to_string(index=False))

long = (pd.MultiIndex.from_product([blks, regs], names=["blk", "p_region"]).to_frame(index=False)
          .merge(cnt, on=["p_region", "blk"], how="left").fillna({"n_new": 0}))
long["ln_n"] = np.log1p(long["n_new"]); long = long.join(RA[["n_inst", "n_fac"]], on="p_region")
long["l_n_inst"] = np.log1p(long["n_inst"]); long["l_n_fac"] = np.log1p(long["n_fac"])
long = long.sort_values(["blk", "p_region"]).reset_index(drop=True)
X = pd.concat([long[["l_n_inst", "l_n_fac"]], pd.get_dummies(long["blk"], prefix="b", drop_first=True).astype(float)], axis=1)
y = long[["ln_n"]].values
for k in (4, 6):
    w = knn_w(k); WB = lps.weights.WSP2W(lps.weights.WSP(sparse.block_diag([w.sparse] * len(blks), format="csr"))); WB.transform = "R"
    ols = spreg.OLS(y, X.values, w=WB, spat_diag=True, moran=True, name_x=list(X.columns), name_y="ln_n")
    print(f"\n[B2] k={k} OLS 진단: Moran 잔차 p={ols.moran_res[2]:.3f} | LM-lag p={ols.lm_lag[1]:.3f} | LM-error p={ols.lm_error[1]:.3f}")
    m = spreg.ML_Lag(y, X.values, w=WB, name_x=list(X.columns), name_y="ln_n")
    print(f"SAR N={len(long)}, rho={m.rho:.3f}")
    print(pd.DataFrame({"var": m.name_x, "coef": m.betas.ravel(), "p": [z[1] for z in m.z_stat]}).round(3).to_string(index=False))

In [ ]:
#

In [ ]:
import os, json, time, hashlib
import numpy as np, pandas as pd, requests
pd.set_option("display.width", 220)

OUT = "oa_out"; CACHE = os.path.join(OUT, "cache"); BASE = "https://api.openalex.org"
MAILTO = "<REDACTED_EMAIL>"; OA_KEY = os.environ.get("OPENALEX_API_KEY", "")
OA_KEY = "<REDACTED_API_KEY>"
YMIN = globals().get("YMIN", 2006); BLK = globals().get("BLK", 5); NB = 4
TYPES = "type:article|review,is_paratext:false"
ALL = ["FUELCELL", "BATTERY", "VEHICLE", "POWER", "SOLAR", "LIGHT", "THERMAL", "MATERIAL"]
NCALL = 0
os.makedirs(CACHE, exist_ok=True)

def oa_get(path, params, tries=8):
    global NCALL
    fn = os.path.join(CACHE, hashlib.md5((path + json.dumps(params, sort_keys=True)).encode()).hexdigest() + ".json")
    if os.path.exists(fn):
        try: return json.load(open(fn, encoding="utf-8"))
        except json.JSONDecodeError: os.remove(fn)
    assert OA_KEY, "환경변수 OPENALEX_API_KEY를 설정하세요"
    p = dict(params); p["mailto"] = MAILTO; p["api_key"] = OA_KEY; last = None
    for t in range(tries):
        try: r = requests.get(BASE + path, params=p, timeout=120)
        except requests.RequestException as e:
            last = str(e); time.sleep(min(60, 2 ** t)); continue
        if r.status_code == 200:
            NCALL += 1; j = r.json(); json.dump(j, open(fn, "w", encoding="utf-8"), ensure_ascii=False); time.sleep(0.2); return j
        last = f"HTTP {r.status_code}: {r.text[:150]}"
        if r.status_code in (429, 500, 502, 503, 504): time.sleep(min(90, 3 * 2 ** t)); continue
        raise RuntimeError(last)
    raise RuntimeError("재시도 초과: " + str(last))

# ---------- 1. 국가 중심점·지역
try:
    rc = requests.get("https://restcountries.com/v3.1/all", params={"fields": "cca2,latlng,region,subregion"}, timeout=60).json()
    CC = pd.DataFrame([dict(country=x["cca2"], c_lat=x["latlng"][0], c_lon=x["latlng"][1],
                            region=x.get("region"), subregion=x.get("subregion")) for x in rc if x.get("latlng")])
    print("[국가 중심점] REST Countries:", len(CC), "개국")
except Exception as e:
    print("REST Countries 실패 -> 파트너 기관 좌표 평균으로 대체(대국에서 부정확, 지역 더미 없음):", str(e)[:80])
    G0 = pd.read_csv(f"{OUT}/inst_geo.csv")
    CC = G0.groupby("country").agg(c_lat=("lat", "mean"), c_lon=("lon", "mean")).reset_index()
    CC["region"] = "미상"; CC["subregion"] = "미상"

# ---------- 2. 국가별 GX 연구 규모 (블록별)
TM = pd.read_csv(f"{OUT}/oa_topic_domain_checked.csv", dtype={"topic_id": str})
topics = sorted(TM.loc[TM["gx_domain"].isin(ALL), "topic_id"].unique())
chunks = [topics[i:i + 50] for i in range(0, len(topics), 50)]
rows = []
for b in range(NB):
    y0 = YMIN + b * BLK; y1 = y0 + BLK - 1
    for ch in chunks:
        j = oa_get("/works", {"filter": f"primary_topic.id:{'|'.join(ch)},{TYPES},publication_year:{y0}-{y1}",
                              "group_by": "authorships.countries", "per_page": 200})
        g = j.get("group_by", [])
        if len(g) >= 200: print(f"  [주의] 국가 버킷 200 도달(블록 {b}) -> 하위 국가 누락 가능")
        rows += [dict(blk=b, country=x["key"].split("/")[-1], n=x["count"]) for x in g]
MASS = pd.DataFrame(rows).groupby(["country", "blk"], as_index=False)["n"].sum().rename(columns={"n": "mass"})
print("country_mass 행:", len(MASS), "| 블록별 국가 수:", MASS.groupby("blk")["country"].nunique().to_dict(), "| 호출", NCALL)
print("일본 GX 논문 규모(블록별, 참고):", MASS[MASS["country"] == "JP"].set_index("blk")["mass"].to_dict())
CC.to_csv(f"{OUT}/country_centroids.csv", index=False, encoding="utf-8-sig")
MASS.to_csv(f"{OUT}/country_mass.csv", index=False, encoding="utf-8-sig")

In [ ]:
import statsmodels.api as sm, statsmodels.formula.api as smf
CC = pd.read_csv(f"{OUT}/country_centroids.csv"); MASS = pd.read_csv(f"{OUT}/country_mass.csv")

Fo = P[(P["country"] != "JP") & P["country"].notna()].copy()
cn_all = Fo.groupby(["corp_norm", "country", "blk"])["work_id"].nunique().rename("n")
cn_fu = Fo[Fo["itype"] == "education"].groupby(["corp_norm", "country", "blk"])["work_id"].nunique().rename("n_fu")
hq = P.drop_duplicates("corp_norm")[["corp_norm", "hq_lat", "hq_lon"]]

firms = sorted(Fo["corp_norm"].unique())
countries = sorted(set(Fo["country"]) & set(CC["country"]) & set(MASS["country"]))
dropped = sorted(set(Fo["country"]) - set(countries))
print(f"기업 {len(firms)} x 국가 {len(countries)} x 블록 {NB} | 중심점·규모 없어 제외된 국가 {len(dropped)}곳: {dropped[:10]}")

D = pd.MultiIndex.from_product([firms, countries, range(NB)], names=["corp_norm", "country", "blk"]).to_frame(index=False)
D = (D.merge(cn_all.reset_index(), how="left").merge(cn_fu.reset_index(), how="left")
       .merge(hq, on="corp_norm").merge(CC, on="country").merge(MASS, on=["country", "blk"], how="left"))
D[["n", "n_fu"]] = D[["n", "n_fu"]].fillna(0)
D["mass"] = D["mass"].fillna(0)

def hav(lat1, lon1, lat2, lon2):
    p = np.pi / 180
    a = np.sin((lat2 - lat1) * p / 2) ** 2 + np.cos(lat1 * p) * np.cos(lat2 * p) * np.sin((lon2 - lon1) * p / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))
D["dist"] = hav(D["hq_lat"], D["hq_lon"], D["c_lat"], D["c_lon"])
D["ln_d"] = np.log(D["dist"].clip(lower=50)); D["ln_d"] -= D["ln_d"].mean()      # 중심화: 계수 해석 용이
D["ln_mass"] = np.log1p(D["mass"]); D["ln_mass"] -= D["ln_mass"].mean()
D = D.sort_values(["corp_norm", "country", "blk"])
D["lag_n"] = D.groupby(["corp_norm", "country"])["n"].shift(1)
D["ln_lag"] = np.log1p(D["lag_n"])
for b in range(1, NB): D[f"d_b{b}"] = D["ln_d"] * (D["blk"] == b)
D["fb"] = D["corp_norm"].astype(str) + "_" + D["blk"].astype(str)
D["region"] = D["region"].fillna("미상")
print(f"격자 {len(D):,}행 | 0 비율 {(D['n'] == 0).mean():.3f} | 총 협력 논문(기업-국가-블록 합) {int(D['n'].sum()):,}")
print("거리 범위(km):", int(D["dist"].min()), "~", int(D["dist"].max()), "| 국가 간 거리 변이 대비 기업 간 변이(표준편차 km):",
      round(D.groupby("country")["dist"].std().mean(), 1), "vs", round(D.groupby("corp_norm")["dist"].mean().std(), 1))

# ---- PPML 함수: 기업x블록 FE, 국가 군집 표준오차
def ppml(d, y, rhs, title, show=None, cluster="country"):
    d = d[d.groupby(["corp_norm", "blk"])[y].transform("sum") > 0]
    mod = smf.glm(f"{y} ~ {rhs} + C(fb)", data=d, family=sm.families.Poisson(), missing="drop")
    idx = mod.data.row_labels
    m = mod.fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d.loc[idx, cluster])[0]}, maxiter=300)
    keep = [v for v in m.params.index if not v.startswith("C(fb)") and v != "Intercept"]
    if show: keep = [v for v in keep if any(s in v for s in show)]
    print(f"\n{title} | N={int(m.nobs):,}, 기업 {d.loc[idx, 'corp_norm'].nunique()}곳, 국가 {d.loc[idx, 'country'].nunique()}곳, 군집(국가) {d.loc[idx, cluster].nunique()}")
    print(pd.DataFrame({"coef": m.params[keep], "se": m.bse[keep], "p": m.pvalues[keep]}).round(3).to_string())
    return m

DI = "d_b1 + d_b2 + d_b3"
m1 = ppml(D, "n", "ln_d + ln_mass", "[G1] 거리 탄력성 (전체 해외 협력 논문)")
m2 = ppml(D, "n", f"ln_d + {DI} + ln_mass", "[G2] 거리 탄력성의 블록별 변화 (기준=2006-10)")
try:
    w = m2.wald_test("d_b1 = 0, d_b2 = 0, d_b3 = 0", scalar=True)
    print(f"  [G2] 거리x블록 상호작용 공동검정 p = {float(w.pvalue):.3f}")
except Exception as e:
    print("  Wald 생략:", str(e)[:80])
b0 = m2.params["ln_d"]
print("  블록별 거리 탄력성:", {LAB.get(b, b): round(b0 + (m2.params.get(f"d_b{b}", 0) if b else 0), 3) for b in range(NB)})

m3 = ppml(D, "n", f"ln_d + {DI} + ln_mass + C(region)", "[G3] + 대륙 지역 더미 (거리 효과가 지역 내에서 남는가)", show=["ln_d", "d_b", "ln_mass"])
m4 = ppml(D[D["blk"] >= 1], "n", f"ln_d + {DI.replace('d_b1 + ', '')} + ln_mass + ln_lag", "[G4] + 직전 블록 협력(관계 지속)", show=["ln_d", "d_b", "ln_mass", "ln_lag"])
m5 = ppml(D, "n_fu", f"ln_d + {DI} + ln_mass", "[G5] 해외 '대학' 협력만")
top = Fo.groupby("corp_norm")["work_id"].nunique().sort_values(ascending=False).index[:2].tolist()
m6 = ppml(D[~D["corp_norm"].isin(top)], "n", f"ln_d + {DI} + ln_mass", f"[G6] 상위 2개사 제외({top})")
m7 = ppml(D, "n", f"ln_d + {DI} + ln_mass", "[G7] 군집을 기업으로 바꾼 민감도", cluster="corp_norm")

In [ ]:
#